In [21]:
%load_ext autoreload
%autoreload 2

import numpy as np 
import polars as pl
import sys
import os
import ctypes

# Set path (only needed for Windows users to resolve incompatibility from rpy2)
conda_env_path = r"C:\Users\liang\anaconda3\envs\hdc"
r_home = os.path.join(conda_env_path, "lib", "R")
r_bin = os.path.join(r_home, "bin", "x64")
library_bin = os.path.join(conda_env_path, "Library", "bin")

# Set variables
os.environ["R_HOME"] = r_home
os.environ["PATH"] = r_bin + ";" + library_bin + ";" + os.environ["PATH"]

# Load DLLs manually
try:
    if os.path.exists(r_bin):
        os.add_dll_directory(r_bin)
    if os.path.exists(library_bin):
        os.add_dll_directory(library_bin)
    print("DLLs loaded successfully.")
except AttributeError:
    pass

# --- BYPASS RPY2 AUTO-DETECTION ---
# rpy2 crashes trying to find flags we already loaded. We will silence it.
import rpy2.situation

def mock_get_r_flags(r_home, *args):
    # Create a dummy object that returns an empty list of libraries
    class MockFlags:
        L = [] 
    return (MockFlags(),)

# Overwrite the function causing the crash
rpy2.situation.get_r_flags = mock_get_r_flags
print("Patched rpy2 to skip auto-detection.")

sys.path.append('../')
from NeuroHDC.loader import RLoader 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
DLLs loaded successfully.
Patched rpy2 to skip auto-detection.


In [23]:
def preprocess(data,training_window,bin_size,n_states = 5):
    ''' 
        Preprocess the odor data into training
    '''
    odor_data = {}
    rat_name = ['Barat','Buchanan','Mitt','Stella','Superchris']

    for irat in range(5):
        bin_spk =(    
            data[irat]# only use one rat for now 
            # compute relative t
            .with_columns(
                        (1 + 1e3*(pl.col('TimeBin') - pl.col('TimeBin').min().over(['trial_id','y'])))
                        .round()
                        .cast(pl.Int32)
                        .alias('relative_t')
                        )
            # e.g. extract 250-500 after poke in
            .filter(
                (pl.col('relative_t') >= training_window[0]) & 
                (pl.col('relative_t') <  training_window[1]) )
            # bin them
            .with_columns(
                (pl.col('relative_t')//bin_size).alias('bin_id')
            ) 
            # compute the total firing spk
            .group_by('y','seq_id','trial_id','bin_id')
            .agg(
                pl.col("^.*U.*$").sum() 
            )
            .with_columns(
                        (pl.col('y')-1).alias('y'),
                        (pl.col('bin_id')-pl.min('bin_id')).alias('bin_id'))
            .filter(pl.col('y')<n_states)
            .sort('y','seq_id','trial_id','bin_id')) 

        split_trial = [spk[:,4:].to_numpy() for spk in bin_spk.partition_by('trial_id')]
        y = np.concatenate([spk["y"].unique().to_numpy() for spk in bin_spk.partition_by("trial_id")])
        tri = np.concatenate([spk["trial_id"].unique().to_numpy() for spk in bin_spk.partition_by("trial_id")])
        
        split_trial_np = np.array(split_trial) #(trials, T, neurons)
          
        odor_data[irat] = {
            'rat_name': rat_name[irat],
            'binned_spk': split_trial_np,
            'y':y, 
            'trial_id':tri,
            'spk_df':bin_spk}
        
    return odor_data


def preprocess_run(train_data,bin_size):
    ''' 
        Preprocess the running data into test
    '''
    run_data = {}
    rat_name = ['Barat','Buchanan','Mitt','Stella','Superchris']

    for irat in range(5):
        bin_spk = (
                train_data[irat]
                .with_columns(
                        (1e3*(pl.col('TimeBin') - pl.col('TimeBin').min().over(['trial_id','y'])))
                        .round()
                        .cast(pl.Int32)
                        .alias('relative_t')
                        )
                .filter(pl.col('y')==6,pl.col('relative_t')<200) 
                .with_columns(
                            (pl.col('relative_t')//bin_size).alias('bin_id')
                        ) 
                # compute the total firing spk
                .group_by('y','seq_id','trial_id','bin_id')
                .agg(
                    pl.col("^.*U.*$").sum() 
                )
                .sort('y','seq_id','bin_id') 
            )

        
        split_trial = [spk[:,4:].to_numpy() for spk in bin_spk.partition_by('trial_id')]
        y = np.concatenate([spk["y"].unique().to_numpy() for spk in bin_spk.partition_by("trial_id")])
        tri = np.concatenate([spk["trial_id"].unique().to_numpy() for spk in bin_spk.partition_by("trial_id")])
        
        split_trial_np = np.array(split_trial) #(trials, T, neurons)
 
        run_data[irat] = {
            'rat_name': rat_name[irat],
            'binned_spk': split_trial_np,
            'y':y, 
            'trial_id':tri,
            'spk_df':bin_spk}
    return run_data

In [24]:
#---------- load class---------------------------
# NOTE change it to your path!
raw_loader = RLoader(path='../data/OST_r/')

#---------- load raw data---------------------------
# NOTE num_class choice: 4(A-D), 5(A-E), 6(A-E + Running)
train_data = raw_loader.load_train_data()

total rat number for train is:5


In [25]:
#---------- preprocess data---------------------------
# extract the training window, and bin the spike
training_window = (200,600)
my_bin_size = 25

odor_data = preprocess(
    train_data,
    training_window=training_window,
    bin_size=my_bin_size,
    n_states = 4
    )

In [27]:
from NeuroHDC.fn import check_dist

rat_name = ['Barat','Buchanan','Mitt','Stella','Superchris']
for i, rat in enumerate(rat_name):
    print(f"Rat name: {rat}")
    X = odor_data[i]['binned_spk']
    y = odor_data[i]['y']
    print(f"Number of Trials: {X.shape[0]}, number of bins: {X.shape[1]}, number of neurons: {X.shape[2]}")
    
    check_dist(X,y)

rat name: Barat
Number of Trials: 133, number of bins: 16, number of neurons: 92
Raw Euclidean Dist | Same: 13.60 | Diff: 13.66
rat name: Buchanan
Number of Trials: 174, number of bins: 16, number of neurons: 79
Raw Euclidean Dist | Same: 12.07 | Diff: 12.50
rat name: Mitt
Number of Trials: 207, number of bins: 16, number of neurons: 104
Raw Euclidean Dist | Same: 11.44 | Diff: 11.60
rat name: Stella
Number of Trials: 152, number of bins: 16, number of neurons: 49
Raw Euclidean Dist | Same: 12.36 | Diff: 12.62
rat name: Superchris
Number of Trials: 164, number of bins: 16, number of neurons: 46
Raw Euclidean Dist | Same: 11.66 | Diff: 12.23


In [28]:
from pathlib import Path
import pickle

out_path = Path("../data/rat") / f"odor_prep_{training_window}_{my_bin_size}.pickle"
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("wb") as f:
    pickle.dump(odor_data, f, protocol=pickle.HIGHEST_PROTOCOL)

In [29]:
run_data = preprocess_run(
    train_data,bin_size=my_bin_size)

In [32]:
rat_name = ['Barat','Buchanan','Mitt','Stella','Superchris']
for i, rat in enumerate(rat_name):
    print(f"Rat name: {rat}")
    X = run_data[i]['binned_spk']
    y = run_data[i]['y']
    print(f"Number of Trials: {X.shape[0]}, number of bins: {X.shape[1]}, number of neurons: {X.shape[2]}")

rat name: Barat
Number of Trials: 21, number of bins: 8, number of neurons: 92
rat name: Buchanan
Number of Trials: 28, number of bins: 8, number of neurons: 79
rat name: Mitt
Number of Trials: 23, number of bins: 8, number of neurons: 104
rat name: Stella
Number of Trials: 24, number of bins: 8, number of neurons: 49
rat name: Superchris
Number of Trials: 26, number of bins: 8, number of neurons: 46


In [34]:
run_window = (0,200)

out_path = Path("../data/rat") / f"run_prep_{run_window}_{my_bin_size}.pickle"
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("wb") as f:
    pickle.dump(run_data, f, protocol=pickle.HIGHEST_PROTOCOL)